# Generazione campioni: CVAE, cGAN, DDPM


## Setup comune

In [ ]:
!git clone https://github.com/alessiobsc/GenerativeAI---Face-Generation-on-CelebA.git
!pip -q install gdown


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch torchvision matplotlib

## VAE

In [ ]:
%cd /content/GenerativeAI---Face-Generation-on-CelebA/vae
!mkdir -p weights
!ln -sf "/content/drive/MyDrive/Generative AI Project/vae/weights/cvae_weights_epoch83.ckp" weights/cvae_weights_epoch83.ckp
!python generate_samples.py

from IPython.display import Image
Image("samples/cvae_generation_grid.png")

## cGAN

In [ ]:
import glob, torch
from IPython.display import Image, display

DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/Generative AI Project/gan/weights"
CHECKPOINT_NAME   = "cgan_generator_only.pt"
CHECKPOINT_PATH   = f"{DRIVE_WEIGHTS_DIR}/{CHECKPOINT_NAME}"
N_PER_CLASS = 4
SEED        = 123
OUT_DIR     = "/content/results_gan"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

%cd /content/GenerativeAI---Face-Generation-on-CelebA/gan
!python generate_cgan.py --checkpoint "{CHECKPOINT_PATH}" --out_dir {OUT_DIR} --n_per_class {N_PER_CLASS} --seed {SEED} --device {DEVICE}

pngs = sorted(glob.glob(f"{OUT_DIR}/*.png"))
display(Image(filename=pngs[-1])) if pngs else print("Nessuna immagine trovata in", OUT_DIR)

## DDPM

In [ ]:
import os, re, glob
from IPython.display import Image, display

CHECKPOINT_NAME = "ckpt_e500.pt"
CHECKPOINT_PATH = f"/content/drive/MyDrive/Generative AI Project/ddpm/weights/{CHECKPOINT_NAME}"
OUT_DIR = "/content/results_ddpm"
SCRIPT  = "/content/GenerativeAI---Face-Generation-on-CelebA/ddpm/test_ddpm.py"

if not os.path.isfile(CHECKPOINT_PATH):
    for root, _, files in os.walk("/content/drive/MyDrive"):
        if CHECKPOINT_NAME in files:
            CHECKPOINT_PATH = os.path.join(root, CHECKPOINT_NAME)
            break
    else:
        raise FileNotFoundError(f"{CHECKPOINT_NAME} non trovato in Google Drive")
print("Checkpoint:", CHECKPOINT_PATH)

src = open(SCRIPT, encoding="utf-8").read()
src = re.sub(r"^CHECKPOINT_PATH = .*$", lambda m: f"CHECKPOINT_PATH = {CHECKPOINT_PATH!r}", src, flags=re.M)
src = re.sub(r"^OUTPUT_DIR = .*$", lambda m: f"OUTPUT_DIR = {OUT_DIR!r}", src, flags=re.M)
open("/content/test_ddpm_colab.py", "w", encoding="utf-8").write(src)

!python /content/test_ddpm_colab.py

pngs = sorted(glob.glob(f"{OUT_DIR}/*.png"))
display(Image(filename=pngs[-1])) if pngs else print("Nessuna immagine trovata in", OUT_DIR)